## Validate the outcomes

### Original source data

In [4]:
import pandas as pd

vtcoms = pd.read_csv("../data/zepto_v1.csv")
print("Source data shape:",vtcoms.shape)

Source data shape: (3732, 9)


### PostgreSQL analytical result

In [5]:
sql_result = pd.read_csv("../outputs/stock_out_by_category.csv")
print("SQL result: ")
print(sql_result)

SQL result: 
                 category  total_products  out_of_stock_products  \
0                Biscuits             147                     42   
1               Beverages             129                     28   
2   Dairy, Bread & Batter             129                     28   
3      Meats, Fish & Eggs              63                     12   
4        Health & Hygiene              97                     13   
5                Munchies             514                     64   
6      Cooking Essentials             514                     64   
7    Ice Cream & Desserts             388                     45   
8    Chocolates & Candies             388                     45   
9           Packaged Food             388                     45   
10        Home & Cleaning             194                     19   
11    Fruits & Vegetables              93                      6   
12          Personal Care             344                     21   
13            Paan Corner          

### Independently calculate stock-out rate

In [13]:
python_result = (
    vtcoms.groupby("Category").agg(
        total_products=("name", "count"), 
        out_of_stock_products=("outOfStock", "sum")
    ).reset_index()
)

python_result["stockout_rate"] = (
    python_result["out_of_stock_products"] /
    python_result["total_products"] * 100
).round(2)

python_result = python_result.rename(
    columns={"Category": "category"}
)

sql_result = (
    sql_result
    .sort_values("category")
    .reset_index(drop=True)
)

python_result = (
    python_result
    .sort_values("category")
    .reset_index(drop=True)
)

# Compare
comparison = sql_result.merge(
    python_result,
    on="category",
    suffixes=("_sql", "_python")
)



# Calculate difference
comparison["rate_difference"] = (
    comparison["stockout_rate_sql"]
    - comparison["stockout_rate_python"]
).abs()


print("\nComparison:")
print(
    comparison[
        [
            "category",
            "stockout_rate_sql",
            "stockout_rate_python",
            "rate_difference"
        ]
    ]
)

if (comparison["rate_difference"] < 0.01).all():
    print("\nVALIDATION PASSED")
    print(
        "SQL stock-out rates match the independent "
        "Python calculation."
    )
else:
    print("\nVALIDATION FAILED")
    print(
        "There is a difference between SQL and Python results."
    )


Comparison:
                 category  stockout_rate_sql  stockout_rate_python  \
0               Beverages              21.71                 21.71   
1                Biscuits              28.57                 28.57   
2    Chocolates & Candies              11.60                 11.60   
3      Cooking Essentials              12.45                 12.45   
4   Dairy, Bread & Batter              21.71                 21.71   
5     Fruits & Vegetables               6.45                  6.45   
6        Health & Hygiene              13.40                 13.40   
7         Home & Cleaning               9.79                  9.79   
8    Ice Cream & Desserts              11.60                 11.60   
9      Meats, Fish & Eggs              19.05                 19.05   
10               Munchies              12.45                 12.45   
11            Paan Corner               6.10                  6.10   
12          Packaged Food              11.60                 11.60   
13     

### Save validation result

In [14]:
comparison.to_csv(
    "../outputs/validation_stockout.csv",
    index=False
)

print("Validation file saved.")

Validation file saved.


### Category level average discount

In [16]:
sql_discount = pd.read_csv("../outputs/overall_discounts_per_category.csv")

python_discount = (
    df.groupby("Category")
      .agg(
          avg_discount=("discountPercent", "mean")
      )
      .reset_index()
)

python_discount["avg_discount"] = python_discount["avg_discount"].round(2)

python_discount = python_discount.rename(
    columns={"Category": "category"}
)

# Sort
sql_discount = sql_discount.sort_values("category").reset_index(drop=True)
python_discount = python_discount.sort_values("category").reset_index(drop=True)

# Compare
comparison_discount = sql_discount.merge(
    python_discount,
    on="category",
    suffixes=("_sql", "_python")
)

comparison_discount["difference"] = (
    comparison_discount["avg_discount_sql"]
    - comparison_discount["avg_discount_python"]
).abs()

print(comparison_discount)

if (comparison_discount["difference"] < 0.01).all():
    print("DISCOUNT VALIDATION PASSED")
else:
    print("DISCOUNT VALIDATION FAILED")

                 category  avg_discount_sql  min_discount  max_discount  \
0               Beverages              7.16           0.0          50.0   
1                Biscuits              8.24           0.0          51.0   
2    Chocolates & Candies              8.32           0.0          50.0   
3      Cooking Essentials              7.16           0.0          50.0   
4   Dairy, Bread & Batter              7.16           0.0          50.0   
5     Fruits & Vegetables             15.46           4.0          23.0   
6        Health & Hygiene              8.05           0.0          50.0   
7         Home & Cleaning              5.68           0.0          18.0   
8    Ice Cream & Desserts              8.32           0.0          50.0   
9      Meats, Fish & Eggs             11.03           0.0          50.0   
10               Munchies              7.16           0.0          50.0   
11            Paan Corner              6.25           0.0          45.0   
12          Packaged Food